In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import numpy as np
import os

In [2]:
SITE_INFO = [
    "site_id", "observation_hour", "station_name"
]

STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg"
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

TARGET = "streamflow_cfs_mean" 

In [3]:
lp3_df = pl.read_csv('../../src/static/lp3_results.csv')
lp3_sites_df = lp3_df.filter(pl.col("flood_severity_score") >= 2.18)['site_id'].to_list()
lp3_sites_df = [str(s).zfill(8) for s in lp3_sites_df]

In [4]:
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

LOAD_COLS = SITE_INFO + STATIC_FEATURES + DYNAMIC_FEATURES

df = (
    pl.scan_parquet(f"{artifact_dir}/flood_model.parquet")
    .select(LOAD_COLS)
    .filter(pl.col("site_id").is_in(lp3_sites_df))
    .filter(pl.col("observation_hour") <= pl.lit("2024-08-01").str.to_datetime())
    .collect()
)

print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total sites: {df['site_id'].n_unique()}")
print(f"Date range: {df['observation_hour'].min()} to {df['observation_hour'].max()}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset:latest', 4549.24MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.4 (11205.0MB/s)


Full dataset: 4,639,150 rows x 31 columns
Total sites: 37
Date range: 2007-10-28 05:00:00 to 2024-07-12 04:00:00


In [5]:
df = df.with_columns(
    pl.when(pl.col("streamflow_cfs_mean") < 0).then(None).otherwise(pl.col("streamflow_cfs_mean")).alias("streamflow_cfs_mean"),
)
print("Cleaned bad values: streamflow < 0 → null")

# Compute streamflow statistics per site
site_stats = df.group_by("site_id").agg(
    pl.col("streamflow_cfs_mean").mean().alias("streamflow_mean"),
    pl.len().alias("total_rows"),
    pl.col("streamflow_cfs_mean").null_count().alias("streamflow_nulls"),
    pl.col("gage_height_ft_mean").null_count().alias("gage_height_nulls"),
)

# Compute coefficient of variation and null rates
site_stats = site_stats.with_columns(
    (pl.col("streamflow_nulls") / pl.col("total_rows") * 100).alias("streamflow_null_pct"),
    (pl.col("gage_height_nulls") / pl.col("total_rows") * 100).alias("gage_height_null_pct"),

)

print(f"Total sites: {len(site_stats)}")
print(f"Sites with 100% null streamflow (no data): {site_stats.filter(pl.col('streamflow_null_pct') == 100).shape[0]}")
print(f"Sites with zero mean streamflow: {site_stats.filter(pl.col('streamflow_mean') == 0).shape[0]}")

Cleaned bad values: streamflow < 0 → null
Total sites: 37
Sites with 100% null streamflow (no data): 0
Sites with zero mean streamflow: 0


In [6]:
# Distribution of null rates
null_data = site_stats.to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Streamflow Null %", "Gage Height Null %"))

fig.add_trace(
    go.Histogram(x=null_data["streamflow_null_pct"], nbinsx=50, name="Streamflow", marker_color="steelblue"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=null_data["gage_height_null_pct"], nbinsx=50, name="Gage Height", marker_color="darkorange"),
    row=1, col=2
)


fig.update_layout(
    title="Distribution of Null Rates Across Sites",
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(title_text="Percents (%)", showline=True, linecolor="black", linewidth=1, mirror=True)
fig.update_yaxes(title_text="Number of Sites",
                 showline=True, linecolor="black", linewidth=1, mirror=True,
                 showgrid=True, gridcolor="lightgrey", gridwidth=0.5)
fig.show()

# Summary
print(f"Sites with 0% streamflow nulls:    {len(null_data[null_data['streamflow_null_pct'] == 0])}")
print(f"Sites with <20% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] < 20])}")
print(f"Sites with >50% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] > 50])}")
print(f"Sites with 100% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] == 100])}")
print()
print(f"Sites with 0% gage height nulls:   {len(null_data[null_data['gage_height_null_pct'] == 0])}")
print(f"Sites with <20% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] < 20])}")
print(f"Sites with >50% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] > 50])}")
print(f"Sites with 100% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] == 100])}")

Sites with 0% streamflow nulls:    1
Sites with <20% streamflow nulls:  35
Sites with >50% streamflow nulls:  0
Sites with 100% streamflow nulls:  0

Sites with 0% gage height nulls:   0
Sites with <20% gage height nulls: 24
Sites with >50% gage height nulls: 12
Sites with 100% gage height nulls: 1


In [10]:
PROJECT = "flood-forecasting"
ARTIFACT_NAME = "flood-dataset-top30"
PARQUET_NAME = "flood_model_top30.parquet"

df.write_parquet(PARQUET_NAME)

run = wandb.init(
   project=PROJECT,
    job_type="dataset-sync",
    config={
        "source": "site_filtering.ipynb",
        "sites": "missouri_only",
        "row_count": int(df.shape[0]),
        "site_count": int(df["site_id"].n_unique()),
    },
)

artifact = wandb.Artifact(
    name=ARTIFACT_NAME,
    type="dataset",
    description="ML-ready flood_model subset for 37 sites and 31 columns (hourly).",
)

artifact.add_file(PARQUET_NAME)
run.log_artifact(artifact, aliases=["latest"])
run.finish()

wandb: Currently logged in as: leclainffalonzosacha (connorjsmith28-rice-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
run = wandb.init(
    project=PROJECT,
    job_type="dataset-sync",
    config={
        "source": "site_filtering.ipynb",
        "sites": "lp3_filtered",
        "row_count": int(df.shape[0]),
        "site_count": int(df["site_id"].n_unique()),
        "date_range_start": str(df["observation_hour"].min()),
        "date_range_end": str(df["observation_hour"].max()),
        "cutoff_date": "2024-08-01",
        "filter": "flood_severity_score >= 1.25",
        "columns": LOAD_COLS,
        "static_features": STATIC_FEATURES,
        "dynamic_features": DYNAMIC_FEATURES,
    },
)

artifact = wandb.Artifact(
    name=ARTIFACT_NAME,
    type="dataset",
    description=f"ML-ready flood_model subset: {df['site_id'].n_unique()} sites, {df.shape[1]} cols, hourly, filtered by LP3 flood_severity_score >= 1.25, cutoff 2024-08-01.",
    metadata={
        "row_count": int(df.shape[0]),
        "site_count": int(df["site_id"].n_unique()),
        "site_ids": lp3_sites_df,
        "date_start": str(df["observation_hour"].min()),
        "date_end": str(df["observation_hour"].max()),
        "columns": LOAD_COLS,
        "filter": "flood_severity_score >= 1.25",
        "source_artifact": "flood-forecasting/flood-dataset:latest",
    },
)

artifact.add_file(PARQUET_NAME)
run.log_artifact(artifact, aliases=["latest"])
run.finish()

In [12]:
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset-top30:latest")
import json
print(json.dumps(artifact.metadata, indent=2))

{
  "filter": "flood_severity_score >= 1.25",
  "columns": [
    "site_id",
    "observation_hour",
    "station_name",
    "longitude",
    "latitude",
    "DRAIN_SQKM",
    "artificial_path_pct",
    "wb5100_ann_mm",
    "snw_pc_syr",
    "snow_ice_nlcd06",
    "barren_nlcd06",
    "mains100_plant",
    "hga",
    "hgc",
    "bulk_density_avg",
    "elev_max_m",
    "aspect_deg",
    "streamflow_cfs_mean",
    "streamflow_cfs_max",
    "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction"
  ],
  "date_end": "2024-07-12 04:00:00",
  "site_ids": [
    "06911000",
    "06401500",
    "06870300",
    "06915000",
    "06911490",
    "06864000",
    "06869500",
    "06342260",
    "06440200",
    "06784000",
    "06912500",
    "06821